# Lab 19-06 · Where the Power Goes

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import power as pw

## Part 1 · Switching charge

*Predict* the charge and energy drawn from a 1.2 V supply while an inverter charges 10 fF, and how much is stored at the end.

In [2]:
C, VDD = 10e-15, 1.2
print(f"charge {C * VDD * 1e15:.1f} fC, energy drawn {C * VDD ** 2 * 1e15:.1f} fJ, stored {0.5 * C * VDD ** 2 * 1e15:.1f} fJ (ngspice: power.cir, circuit A)")
print("dynamic power of Example 19-06.1:", pw.dynamic_power(0.1, 1e-9, 0.8, 2e9), "W")

charge 12.0 fC, energy drawn 14.4 fJ, stored 7.2 fJ (ngspice: power.cir, circuit A)
dynamic power of Example 19-06.1: 0.12800000000000006 W


## Part 2 · Short-circuit charge

*Predict* the short-circuit charge per period for 50 ps ramps, and how it scales with ramp time.

In [3]:
for tau in (25e-12, 50e-12, 100e-12, 200e-12):
    q = pw.short_circuit_charge_numeric(200e-6, 1.2, 0.35, tau)
    p = pw.short_circuit_power_veendrick(200e-6, 1.2, 0.35, tau, 1e-9)
    print(f"ramp {tau * 1e12:5.0f} ps: {2 * q * 1e15:.4f} fC per period, {p * 1e6:.4f} uW (Veendrick)")

ramp    25 ps: 0.0434 fC per period, 0.0521 uW (Veendrick)
ramp    50 ps: 0.0868 fC per period, 0.1042 uW (Veendrick)
ramp   100 ps: 0.1736 fC per period, 0.2083 uW (Veendrick)
ramp   200 ps: 0.3472 fC per period, 0.4167 uW (Veendrick)


## Part 3 · Leakage and the stack effect

*Predict* how leakage depends on threshold shift, and how the stack effect depends on DIBL.

In [4]:
for dvt in (0.05, 0.1, 0.2):
    print(f"threshold lowered {dvt * 1e3:.0f} mV: leakage x {pw.leakage_factor_for_vt_shift(dvt, 90):.1f}")
etas = np.linspace(0, 0.2, 21)
ratios = [pw.stack_leakage_ratio(1.0, 0.35, 1.3, e)[1] for e in etas]
plt.semilogy(etas, ratios); plt.xlabel("DIBL coefficient eta"); plt.ylabel("single / stack leakage"); plt.show()
for e in (0.0, 0.1, 0.2):
    vx, r = pw.stack_leakage_ratio(1.0, 0.35, 1.3, e)
    print(f"eta = {e}: middle node {vx * 1e3:.1f} mV, reduction {r:.2f}")

threshold lowered 50 mV: leakage x 3.6
threshold lowered 100 mV: leakage x 12.9
threshold lowered 200 mV: leakage x 166.8
eta = 0.0: middle node 20.4 mV, reduction 1.83
eta = 0.1: middle node 84.4 mV, reduction 15.85
eta = 0.2: middle node 143.0 mV, reduction 164.73


## Part 4 · DVFS and parallelism

*Predict* the supply needed for half, a third and a quarter of the frequency (1.0 V, V_T = 0.35 V, alpha = 1.3).

In [5]:
for k in (2, 3, 4):
    v = pw.supply_for_slowdown(k, 1.0, 0.35, 1.3)
    print(f"1/{k} frequency: {v:.3f} V, power {v ** 2 / k:.3f}, energy per op {v ** 2:.3f}")

1/2 frequency: 0.611 V, power 0.187, energy per op 0.373
1/3 frequency: 0.518 V, power 0.090, energy per op 0.269
1/4 frequency: 0.477 V, power 0.057, energy per op 0.227


## Part 5 · The minimum-energy point

*Predict* where energy per operation is smallest for alpha = 0.1, logic depth 20, n = 1.3.

In [6]:
v = np.linspace(0.05, 0.5, 400)
e = [pw.energy_per_op(x, 0.1, 20, 1.3) for x in v]
vm = pw.minimum_energy_supply(0.1, 20, 1.3)
plt.figure(); plt.plot(v * 1e3, np.array(e) / pw.energy_per_op(vm, 0.1, 20, 1.3)); plt.ylim(0, 4); plt.xlabel("supply (mV)"); plt.ylabel("energy / minimum"); plt.show()
x = vm / (1.3 * pw.VT_300K)
print(f"minimum at {vm * 1e3:.1f} mV (x = {x:.2f}); leakage share {2 / x:.1%}")
for a, d in ((0.05, 20), (0.1, 50)):
    print(f"alpha {a}, depth {d}: {pw.minimum_energy_supply(a, d, 1.3) * 1e3:.1f} mV")

minimum at 201.3 mV (x = 5.99); leakage share 33.4%
alpha 0.05, depth 20: 231.4 mV
alpha 0.1, depth 50: 240.7 mV


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).